# 🚀 Huấn Luyện PhoBERT-BiLSTM-CRF Trên Google Colab (GPU Tesla T4)
**Đề tài:** Nhận diện chuỗi ngôn ngữ xúc phạm tiếng Việt (ViHOS)  
**Người phụ trách:** Hoàng Võ Minh Tuấn (Trainer - 26410146)  
**Môi trường:** GPU Tesla T4 (16GB VRAM), PyTorch, HuggingFace Transformers, PyTorch-CRF, Seqeval  
**GitHub Repo chính thức:** [https://github.com/barackvn/NLP](https://github.com/barackvn/NLP)


## Bước 1: Kiểm tra cấu hình phần cứng GPU Tesla T4
Đảm bảo bạn đã chọn **Runtime -> Change runtime type -> T4 GPU** trước khi chạy!

In [ ]:
!nvidia-smi


## Bước 2: Tải Mã Nguồn & Dữ Liệu từ GitHub Chính Thức
Toàn bộ code, dữ liệu 3 tập Train-Dev-Test (11.056 câu) đã sẵn sàng trong repository.

In [ ]:
import os
import shutil

# Nếu đã clone trước đó thì xóa để kéo bản mới nhất, hoặc cd vào
%cd /content
if os.path.exists('/content/NLP'):
    !rm -rf /content/NLP

# Clone repository chính thức của nhóm
!git clone https://github.com/barackvn/NLP.git
%cd /content/NLP

print("✅ Thư mục làm việc hiện tại:")
!pwd
!ls -la


## Bước 3: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q transformers pyvi pytorch-crf seqeval accelerate


## Bước 4: Kết nối Google Drive để sao lưu trọng số (Tùy chọn an toàn)
Kết nối Google Drive giúp lưu vĩnh viễn file checkpoint phòng trường hợp mất mạng hoặc hết phiên Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ViHOS_Checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✅ Thư mục sao lưu Drive: {DRIVE_DIR}")


## Bước 5: Huấn Luyện Mô Hình Đề Xuất (PhoBERT-BiLSTM-CRF)
- **Epochs:** 5  
- **Batch size:** 16  
- **Differential LR:** PhoBERT `2e-5`, BiLSTM-CRF Head `1e-3`  
- **Early Stopping:** patience = 3 (theo dõi Span-F1 trên tập DEV)
- **Thời gian ước tính:** ~15–18 phút trên GPU T4


In [ ]:
!python -m src.train \
    --model_type phobert_bilstm_crf \
    --train_path data/processed/train.json \
    --dev_path data/processed/dev.json \
    --save_path checkpoints/best_phobert_bilstm_crf.pt \
    --epochs 5 \
    --batch_size 16 \
    --lr_phobert 2e-5 \
    --lr_head 1e-3 \
    --max_length 128 \
    --patience 3

# Sao lưu file .pt sang Google Drive
if os.path.exists('checkpoints/best_phobert_bilstm_crf.pt'):
    !cp checkpoints/best_phobert_bilstm_crf.pt /content/drive/MyDrive/ViHOS_Checkpoints/
    print("🎉 ĐÃ LƯU CHECKPOINT VÀO GOOGLE DRIVE THÀNH CÔNG!")


## Bước 6: Tải Trực Tiếp Checkpoint Về Máy Tính Cá Nhân
Bấm chạy cell này để tải file `best_phobert_bilstm_crf.pt` (~540MB) về máy tính qua trình duyệt.  
👉 Sau khi tải về, bạn chỉ cần chép file này vào thư mục `Doan/checkpoints/` trên máy local là Web App và Notebook 03 sẽ tự động nhận diện AI Thật 100%!

In [ ]:
from google.colab import files
if os.path.exists('checkpoints/best_phobert_bilstm_crf.pt'):
    files.download('checkpoints/best_phobert_bilstm_crf.pt')
else:
    print("Chưa tìm thấy file checkpoint. Vui lòng kiểm tra lại log huấn luyện ở Bước 5.")


## Bước 7 (Tùy chọn): Huấn Luyện 2 Mô Hình Đối Chứng (Ablation Baselines)
Dành cho bạn Tuấn và Thịnh để thu thập số liệu đối chứng cho Slide 10-15:
1. `baseline_phobert_linear.pt` (Baseline gốc của Thầy Đặng Văn Thìn)
2. `baseline_phobert_crf.pt` (Mô hình bóc tách không dùng BiLSTM)


In [ ]:
# 1. Baseline Thầy: PhoBERT-Linear
!python -m src.train \
    --model_type phobert_linear \
    --train_path data/processed/train.json \
    --dev_path data/processed/dev.json \
    --save_path /content/drive/MyDrive/ViHOS_Checkpoints/baseline_phobert_linear.pt \
    --epochs 5 \
    --batch_size 16 \
    --lr_phobert 2e-5 \
    --lr_head 1e-3

# 2. Baseline bóc tách: PhoBERT-CRF
!python -m src.train \
    --model_type phobert_crf \
    --train_path data/processed/train.json \
    --dev_path data/processed/dev.json \
    --save_path /content/drive/MyDrive/ViHOS_Checkpoints/baseline_phobert_crf.pt \
    --epochs 5 \
    --batch_size 16 \
    --lr_phobert 2e-5 \
    --lr_head 1e-3
